In [ ]:
# Stage 1 - Data Acquisition
# **Goal:** Pull real SEC 10-Q filings and extract the MD&A narrative section for NLP analysis.

# **What this notebook does:**
# - Downloads JPMorgan Chase 10-Q filings via `sec-edgar-downloader`
# - Navigates the SEC SGML bundle to isolate the primary HTML document
# - Extracts the MD&A section using regex boundary matching
# - Saves clean JSON output to `outputs/` for Stage 2

# **Output:** `outputs/jpm_mda_2025Q2.json` — 597,528 characters of management narrative



In [4]:
from sec_edgar_downloader import Downloader
import os
import pathlib

DATA_DIR = pathlib.Path("../data/filings")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Filing directory: {DATA_DIR.resolve()}")

Filing directory: C:\Users\Ozoha\nlp_finance_sentiment\data\filings


In [5]:
dl = Downloader("GEM", "ozohaustinn@gmail.com", str(DATA_DIR))

dl.get("10-Q", "JPM", limit = 3)

print("Download complete. Check data/filings/")

Download complete. Check data/filings/


In [7]:
filings_path = pathlib.Path("../data/filings/sec-edgar-filings/JPM/10-Q")

for filing_folder in sorted(filings_path.iterdir()):
    files = list(filing_folder.iterdir())
    print(f"\n{filing_folder.name}")
    for f in files:
        print(f" {f.name} ({f.stat().st_size / 1024:.1f} KB)")


0000019617-25-000615
 full-submission.txt (49141.7 KB)

0001628280-25-048859
 full-submission.txt (95560.0 KB)

0001628280-26-029344
 full-submission.txt (42266.8 KB)


In [11]:
latest_folder = sorted(filings_path.iterdir())[0]
filing_file = latest_folder / "full-submission.txt"

with open(filing_file, "r", encoding="utf-8", errors="ignore") as f:
    raw_text = f.read()

print(f"File size: {len(raw_text):,} characters")
print("\nSample (chars 2000–3000):")
print(raw_text[2000:3000])


File size: 50,321,066 characters

Sample (chars 2000–3000):
L" xmlns:country="http://xbrl.sec.gov/country/2025" xmlns:stpr="http://xbrl.sec.gov/stpr/2025" xmlns:xbrli="http://www.xbrl.org/2003/instance" xmlns:srt="http://fasb.org/srt/2025" xmlns:ecd="http://xbrl.sec.gov/ecd/2025" xmlns="http://www.w3.org/1999/xhtml" xmlns:dei="http://xbrl.sec.gov/dei/2025" xmlns:us-gaap="http://fasb.org/us-gaap/2025" xmlns:utr="http://www.xbrl.org/2009/utr" xmlns:link="http://www.xbrl.org/2003/linkbase" xmlns:jpm="http://www.jpmorganchase.com/20250630" xml:lang="en-US"><head><meta http-equiv="Content-Type" content="text/html"/>


<title>jpm-20250630</title></head><body><div style="display:none"><ix:header><ix:hidden><ix:nonNumeric contextRef="c-1" name="dei:EntityCentralIndexKey" id="f-50">0000019617</ix:nonNumeric><ix:nonNumeric contextRef="c-1" name="dei:AmendmentFlag" format="ixt:fixed-false" id="f-51">FALSE</ix:nonNumeric><ix:nonNumeric contextRef="c-1" name="dei:DocumentFiscalYearFocus" id="f-52">

In [15]:
import re

# Find all document blocks and their types
doc_pattern = re.finditer(r'<DOCUMENT>(.*?)</DOCUMENT>', raw_text, re.DOTALL)

documents = []
for doc in doc_pattern:
    content = doc.group(1)
    type_match = re.search(r'<TYPE>(.*?)\n', content)
    sequence_match = re.search(r'<SEQUENCE>(.*?)\n', content)
    filename_match = re.search(r'<FILENAME>(.*?)\n', content)
    doc_type = type_match.group(1).strip() if type_match else "UNKNOWN"
    sequence = sequence_match.group(1).strip() if sequence_match else "?"
    filename = filename_match.group(1).strip() if filename_match else "?"
    documents.append({
        "sequence": sequence,
        "type": doc_type,
        "filename": filename,
        "length": len(content)
    })

for d in documents[:20]:  # show first 20
    print(f"  [{d['sequence']}] {d['type']:<30} {d['filename']:<40} {d['length']:>10,} chars")

  [1] 10-Q                           jpm-20250630.htm                         11,426,834 chars
  [2] EX-15                          corpq22025exhibit15.htm                       6,855 chars
  [3] EX-22                          corpq22025exhibit22.htm                       3,599 chars
  [4] EX-31.1                        corpq22025exhibit311.htm                      8,140 chars
  [5] EX-31.2                        corpq22025exhibit312.htm                      8,108 chars
  [6] EX-32                          corpq22025exhibit32.htm                       9,882 chars
  [7] EX-101.SCH                     jpm-20250630.xsd                            219,290 chars
  [8] EX-101.CAL                     jpm-20250630_cal.xml                        266,912 chars
  [9] EX-101.DEF                     jpm-20250630_def.xml                      1,327,330 chars
  [10] EX-101.LAB                     jpm-20250630_lab.xml                      2,069,720 chars
  [11] EX-101.PRE                     jpm-2025063

In [16]:
# Extract the sequence [1] document — the actual 10-Q
doc_blocks = re.findall(r'<DOCUMENT>(.*?)</DOCUMENT>', raw_text, re.DOTALL)

target_doc = None
for doc in doc_blocks:
    type_match = re.search(r'<TYPE>(.*?)\n', doc)
    if type_match and type_match.group(1).strip() == "10-Q":
        target_doc = doc
        break

# Extract the HTML content from within it
html_match = re.search(r'<TEXT>(.*?)</TEXT>', target_doc, re.DOTALL)
htm_content = html_match.group(1)

print(f"10-Q HTML extracted: {len(htm_content):,} characters")

10-Q HTML extracted: 11,426,751 characters


In [17]:
from bs4 import BeautifulSoup
import re

soup = BeautifulSoup(htm_content, "html.parser")
full_text = soup.get_text(separator=" ", strip=True)
full_text = re.sub(r'\s+', ' ', full_text)

print(f"Full clean text: {len(full_text):,} characters")

# Locate MD&A section — standard SEC heading
mda_pattern = re.search(
    r"(management.{0,10}s discussion and analysis)(.*?)(quantitative and qualitative disclosures about market risk)",
    full_text,
    re.IGNORECASE | re.DOTALL
)

if mda_pattern:
    mda_text = mda_pattern.group(2)
    print(f"\nMD&A section found: {len(mda_text):,} characters")
    print("\nFirst 1500 chars of MD&A:")
    print(mda_text[:1500])
else:
    print("MD&A pattern not matched — printing headings to inspect:")
    # Fallback: show all text between 50k-100k chars to find structure
    print(full_text[50000:51000])

Full clean text: 838,308 characters

MD&A section found: 734 characters

First 1500 chars of MD&A:
 of Financial Condition and Results of Operations. Consolidated Financial Highlights 3 Introduction 4 Executive Overview 5 Consolidated Results of Operations 9 Consolidated Balance Sheets and Cash Flows Analysis 14 Explanation and Reconciliation of the Firm’s Use of Non-GAAP Financial Measures 17 Business Segment & Corporate Results 19 Firmwide Risk Management 42 Capital Risk Management 43 Liquidity Risk Management 50 Consumer Credit Portfolio 60 Wholesale Credit Portfolio 64 Allowance for Credit Losses 73 Investment Portfolio Risk Management 76 Market Risk Management 77 Country Risk Management 84 Critical Accounting Estimates Used by the Firm 85 Accounting and Reporting Developments 89 Forward-Looking Statements 90 Item 3. 


In [18]:
# Find ALL occurrences of the MD&A pattern — TOC gives a short match, real section is long
matches = list(re.finditer(
    r"management.{0,10}s discussion and analysis(.*?)quantitative and qualitative disclosures about market risk",
    full_text,
    re.IGNORECASE | re.DOTALL
))

print(f"Total matches found: {len(matches)}")
for i, m in enumerate(matches):
    print(f"  Match {i+1}: {len(m.group(1)):,} characters")

Total matches found: 3
  Match 1: 734 characters
  Match 2: 597,530 characters
  Match 3: 77 characters


In [19]:
# The real MD&A is by far the longest match
mda_text = max(matches, key=lambda m: len(m.group(1))).group(1)

# Clean it up
mda_text = re.sub(r'\s+', ' ', mda_text).strip()

print(f"MD&A body: {len(mda_text):,} characters")
print("\nFirst 2000 chars:")
print(mda_text[:2000])

MD&A body: 597,528 characters

First 2000 chars:
of the financial condition and results of operations (“MD&A”) of JPMorgan Chase & Co. (“JPMorganChase” or the “Firm”) for the second quarter of 2025. This Quarterly Report on Form 10-Q for the second quarter of 2025 (“Form 10-Q”) should be read together with JPMorganChase’s Annual Report on Form 10-K for the year ended December 31, 2024 (“2024 Form 10-K”). Refer to the Glossary of terms and acronyms and line of business metrics on pages 192-200 for definitions of terms and acronyms used throughout this Form 10-Q. This Form 10-Q contains forward-looking statements within the meaning of the Private Securities Litigation Reform Act of 1995. These forward-looking statements are based on the current beliefs and expectations of JPMorganChase’s management, speak only as of the date of this Form 10-Q and are subject to significant risks and uncertainties. Refer to Forward-looking Statements on page 90 of this Form 10-Q and Part I, Item 1A, Risk 

In [20]:
import json
from datetime import datetime

# Save the raw MD&A text for Stage 2
output = {
    "company": "JPMorgan Chase & Co.",
    "ticker": "JPM",
    "filing_type": "10-Q",
    "period": "2025-06-30",
    "extracted_at": datetime.now().isoformat(),
    "mda_char_count": len(mda_text),
    "mda_text": mda_text
}

output_path = pathlib.Path("../outputs/jpm_mda_2025Q2.json")
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"Saved: {output_path.resolve()}")
print(f"MD&A length: {len(mda_text):,} characters")
print("\nStage 1 complete.")

Saved: C:\Users\Ozoha\nlp_finance_sentiment\outputs\jpm_mda_2025Q2.json
MD&A length: 597,528 characters

Stage 1 complete.
